# (Generalized) coordination numbers in pySNOW

Coordination is a general and simple property of atoms. In the most essential sense, coordination is a count of the number of neighbours of an atom, as we define the coordination number of the atom $i$ as 
$$
CN_i = \sum_{j\in\mathcal{N}(i)}1
$$
where $j$ runs over the atoms in the neighbourhood of the atom $i$: $\mathcal{N}(i)=\{\mathbf{r}_j:|\mathbf{r}_j-\mathbf{r}_i|<r_{cut}\}$, where $\mathbf{r}_\alpha$ is the position vector of the atom $\alpha$.

Naturally, we need to choose a cutoff $r_{cut}$ to define which atoms are neighbours and which are not. See our tutorial on PDDF to have more information on this.

Some more complete and insightful information can be obtained from so-called generalized coordination numbers (GCNs), which take into account the coordination of neighbours as well to define the coordination of a site. GCNs can be defined for atoms (atop GCN) or for sites generally considered as adsorption sites in atomistic systems (bridge, three-hollow, four-hollow).

$$
formule
$$

See how to compute gcns in pySNOW:

In [1]:
from snow.io.xyz import read_xyz, write_xyz

#read some example data
file = 'tutorial_structures/Au13_ih.xyz'

el, coords = read_xyz(file)

In [2]:
#compute the coordination number and atop generalized coordination number
from snow.descriptors.coordination import coordination_number, agcn_calculator

alat   = 4.099
cutoff = 0.89*alat

cns   = coordination_number(coords, cutoff)
sites, agcns = agcn_calculator(coords, cutoff)

#print the first five computed elements
print('                   coordination number of the first five atoms:',cns[:5])
print('(atop) generalized coordination number of the first five atoms:',agcns[:5])
#print the site corresponding to the first computed value:
print('                                         coordinates of site 0:',sites[0])
#here, this is just the position of the first atom. This will change with other types of GCNs

                   coordination number of the first five atoms: [12.  6.  6.  6.  6.]
(atop) generalized coordination number of the first five atoms: [6.  3.5 3.5 3.5 3.5]
                                         coordinates of site 0: [0. 0. 0.]


In [10]:
#compute some other types of gcns
from snow.descriptors.coordination import bridge_gcn, three_hollow_gcn, four_hollow_gcn
 

#bridge sites are only considered between atoms on the surface of your structure.
#An atom is considered on the surface if its coordination is < thr_cn
#use phantom=True to have your function return the pairs of indexes of atoms making up bridge sites as well
bridge_sites, bridge_pairs, bgcns = bridge_gcn(coords, cutoff, thr_cn = 10)

th_sites, th_gcns = three_hollow_gcn(coords, cutoff, thr_cn = 10)
fh_sites, fh_gcns = four_hollow_gcn(coords, cutoff, thr_cn = 10)

[==================================================] 100.00%
Done three hollow
[=================================================-] 99.88%
Done four hollow


You might want to export your computed gcns, e.g., for visualization with OVITO. To do this, you can write fictitious atoms (e.g., Hydrogen atoms if you do not have any Hydrogen in your system) to store data about bridge/hollow sites and their generalized coordination.

In [18]:
import numpy as np

fake_specie = 'H'

#write bridge sites as H atoms
#with info about their generalized coordination 
#as an extra-info array
fake_elements = el + [fake_specie]*len(bridge_sites)

#for now, we need to convert bridge_sites (a list) to a np.ndarray. We'll fix this in the future
bridge_sites_array = np.asarray(bridge_sites)
print(bridge_sites_array.shape)
print(coords.shape)
print(bridge_sites_array[0])
fake_coords = np.vstack((coords, bridge_sites_array))

extra_properties = np.concatenate((np.zeros(len(el)), bgcns))

write_xyz('tutorial_structures/Au13_ih_bgcns.xyz', fake_elements, fake_coords, additional_data=extra_properties)

(30, 3)
(13, 3)
[ 0.758365 -1.22706  -1.985425]


Add strained!